# YouTube 字幕跳板測試（Colab）

**用途**：驗證 Colab 的 IP 能不能抓 YouTube 字幕（本機 IP 被 429 封鎖時的跳板方案）。

**用法**：
1. 先跑 Cell 1（測試）。看到 `✅ Colab IP 可用` 才有必要跑 Cell 2。
2. 看到 `❌ IpBlocked / RequestBlocked` 就代表 Colab 的資料中心 IP 也被 YouTube 封，這條路不通，關掉即可。
3. Cell 2 會批次抓字幕、打包成 `subtitles.zip` 自動下載。

In [ ]:
# Cell 1: 測試 Colab IP 是否能抓字幕
%pip -q install youtube-transcript-api
from youtube_transcript_api import YouTubeTranscriptApi

TEST_VIDEO = "qZhCeV6XATw"  # Big Technology 訪談，確定有英文字幕
try:
    fetched = YouTubeTranscriptApi().fetch(TEST_VIDEO, languages=["en"])
    print(f"✅ Colab IP 可用！抓到 {len(fetched.snippets)} 段字幕")
except Exception as e:
    print(f"❌ 失敗：{type(e).__name__}")
    print(str(e)[:500])

In [ ]:
# Cell 2: 批次抓字幕並打包下載（已預填 2026-07-09 失敗的 22 部）
import json, os, time, zipfile
from youtube_transcript_api import YouTubeTranscriptApi

VIDEO_IDS = [
    "xpeRVyFFy_Q", "Ca0X4O2t9Mo", "zyaabqbDAPM", "eGhEjHnuQ7o",
    "SP2Aty4dKlE", "0swQlqFwjHQ", "skqYvFqwOUA", "LX8VCpl7rHQ",
    "HQDT_1OXUuU", "eAEYPIgKwpI", "cSIMVYjVF28", "UwxxlTNPjWo",
    "tjGVnjsvi-k", "RNY0p5wL4AA", "NcjBCtqRgtE", "ezdo5kUD2WE",
    "n6btoyTPfh4", "qZhCeV6XATw", "0s6UAstE2Ms", "sedZBKHmkL0",
    "-dAVKb6h-k0", "1rSFXB7rQBw",
]
PREFERRED_LANGS = ["zh-TW", "zh-Hant", "zh", "en"]

os.makedirs("subs", exist_ok=True)
api = YouTubeTranscriptApi()
ok, fail = [], []
for vid in VIDEO_IDS:
    try:
        fetched = api.fetch(vid, languages=PREFERRED_LANGS)
        text = "\n".join(s.text for s in fetched.snippets)
        meta = {"video_id": vid, "language": fetched.language_code}
        with open(f"subs/{vid}.txt", "w", encoding="utf-8") as f:
            f.write(json.dumps(meta, ensure_ascii=False) + "\n" + text)
        ok.append(vid)
        print(f"✓ {vid} ({fetched.language_code}, {len(text)} 字)")
    except Exception as e:
        fail.append(vid)
        print(f"✗ {vid}: {type(e).__name__}")
    time.sleep(3)  # 節流，避免 Colab IP 也被封

print(f"\n完成：{len(ok)} 成功、{len(fail)} 失敗")
if ok:
    with zipfile.ZipFile("subtitles.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for vid in ok:
            z.write(f"subs/{vid}.txt")
    from google.colab import files
    files.download("subtitles.zip")